<a href="https://colab.research.google.com/github/C-yue28/FreeCodeCamp-ML-Projects/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [3]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-05-25 21:24:55--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.1’

book-crossings.zip. 100%[===================>]  24.88M  --.-KB/s    in 0.1s    

2026-05-25 21:24:56 (196 MB/s) - ‘book-crossings.zip.1’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace BX-Books.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace BX-Users.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [11]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [12]:
# add your code here - consider creating a new cell for each section of code

# remove non statistically significant entries

user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

active_users = user_counts[user_counts >= 200].index
active_books = book_counts[book_counts >= 100].index

df_ratings = df_ratings[
    df_ratings['user'].isin(active_users) &
    df_ratings['isbn'].isin(active_books)
]

In [15]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):

  pivot_ratings = df_ratings.pivot_table(
    index='isbn',
    columns='user',
    values='rating'
  ).fillna(0)

  matrix = csr_matrix(pivot_ratings.values)

  model = NearestNeighbors(
      metric='cosine',
      algorithm='brute'
  )

  model.fit(matrix)

  isbn = df_books.loc[
    df_books['title'] == book,
    'isbn'
  ].values[0]

  distances, indices = model.kneighbors(
    pivot_ratings.loc[[isbn]],
    n_neighbors=6
  )

  distances = distances[0]
  indices = indices[0]

  recommended_books = [book]

  isbns = pivot_ratings.index[indices]
  titles = [
      df_books.loc[df_books['isbn'] == isbn, 'title'].values[0]
      for isbn in isbns
  ]
  recommended_books.append([[_title, _dist] for _title, _dist in zip(titles[1:], distances[1:])])
  recommended_books[1].sort(key=lambda x: x[0])
  return recommended_books

books = get_recommends('The Queen of the Damned (Vampire Chronicles (Paperback))')
print(books)

['The Queen of the Damned (Vampire Chronicles (Paperback))', [['Catch 22', np.float32(0.7939835)], ['Interview with the Vampire', np.float32(0.73450685)], ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', np.float32(0.53763384)], ['The Vampire Lestat (Vampire Chronicles, Book II)', np.float32(0.51784116)], ['The Witching Hour (Lives of the Mayfair Witches)', np.float32(0.74486566)]]]


In [16]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['I Know This Much Is True', np.float32(0.7677075)], ["I'll Be Seeing You", np.float32(0.8016211)], ['The Lovely Bones: A Novel', np.float32(0.7234864)], ['The Surgeon', np.float32(0.7699411)], ['The Weight of Water', np.float32(0.77085835)]]]
You passed the challenge! 🎉🎉🎉🎉🎉
